# 19 — Director package: surfaces, tiers, clusters, tables (spec v1.1, build steps 1–2)

Zero solves; runs after **18** has completed the guarded sweeps (14/14). Everything here is the
GUARDED semantics (per-block floors; spec v0.14 applied headline), with the unguarded band carried
only for the T-D2 side-by-side. Writes `analyses/y2y/director_package/`:

- `geotiffs/` — `F_guarded.tif`, `F_unguarded.tif`, `f_guarded_<formulation>.tif` ×14,
  `union_membership_guarded.tif`, `act_tiers_guarded.tif`, `cluster_labels.npz`, `clusters.gpkg`
- `tables/` — `T-D1_cluster_register.csv`, `T-D2_bands.csv` + `T-D2_acts.csv`, `T-D3_scenarios.csv`,
  `tier_achievement.csv` (+ reference), `T-D5_protected_baseline.csv`, `T-D4_ecoregions.csv` (needs `input_data/ecoregions/`; pending otherwise),
  `pooling_check.csv`, `cluster_sensitivity.csv`, `cluster_register_all.csv`, `picks.csv`, `E17_shifts.csv`
- `summary.json` — headline numbers consumed by **20**

Pre-stated procedure (spec, unchanged): threshold ≥0.70 → closing r=1 → 8-connected components →
min 100 km² → scenario clusters minus the Act-1 core (overlap reported) → sensitivity at 0.60/0.80.
Top-k selection for the deck is presentational; the full register ships. Kernel `y2y-geo`.


In [ ]:
import importlib, json, pathlib, sys
import numpy as np
import pandas as pd
import rasterio
from scipy import ndimage

_cands = [p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "config.py").exists()]
assert _cands, "config.py not found above the notebook"
ROOT = _cands[0]
sys.path.insert(0, str(ROOT))
import config, leverage_core as lc, ensemble_core as ec, director_core as dc
for _m in (config, lc, ec, dc):
    importlib.reload(_m)

ALLOW_PARTIAL = False   # DEV ONLY: smoke-run on the S0/S4 artifacts from 16 (outputs go to _smoke/)
PKG = dc.ensure_dirs(dc.PKG / "_smoke" if ALLOW_PARTIAL else dc.PKG)
GEO, TAB = PKG / "geotiffs", PKG / "tables"
SPEC = dc.SPEC
G = dc.grid()
MAN = pd.read_csv(SPEC / "manifest.csv")
assert len(MAN) == 14
L = dc.load_guarded(G, MAN, allow_partial=ALLOW_PARTIAL)
FORMS = L.forms
THR = dc.FREQ_THR
SUMMARY = dict(n_formulations=len(FORMS), threshold=THR, floor_g=dc.FLOOR_G, min_km2=dc.MIN_KM2,
               partial=ALLOW_PARTIAL, missing=L.missing)

cert = pd.DataFrame(L.cert).T
cert["D_plain"] = pd.Series(L.D_plain); cert["D_guard"] = pd.Series(L.D_guard)
print("\nguarded sweeps (results_log-ready):")
print(cert.to_string(float_format=lambda v: f"{v:.3f}"))
print(f"total guarded solve time {cert.runtime_min.sum() / 60:.1f} h | duplicates {int(cert.dup.sum())} | "
      f"time-limited {int(cert.time_limited.sum())}")
SUMMARY["guard_runtime_h"] = float(cert.runtime_min.sum() / 60)
# guarded MAA spot-check (S0): instrument-robustness of the GUARDED f
for fid, fm in L.f_maa_guard.items():
    fg = L.f_guard[fid]
    corr = float(np.corrcoef(fg[G.disc], fm[G.disc])[0, 1])
    J = dc.jaccard((fg >= THR) & G.disc, (fm >= THR) & G.disc)
    print(f"\nguarded MAA spot-check {fid}: corr(f_guardMGA, f_guardMAA) {corr:.3f} | frequent km2 "
          f"{int((fg[G.disc] >= THR).sum()):,} vs {int((fm[G.disc] >= THR).sum()):,} | tier Jaccard {J:.3f}")
    SUMMARY["maa_guard_spotcheck"] = dict(formulation=fid, corr=corr, tier_jaccard=J,
                                          freq_km2_mga=int((fg[G.disc] >= THR).sum()),
                                          freq_km2_maa=int((fm[G.disc] >= THR).sum()))


In [ ]:
# ---- surfaces: guarded F (deliverable) + unguarded F (side-by-side) + union membership ----------
Fg = dc.ensemble(L.f_guard, FORMS)
Fp = dc.ensemble(L.f_plain, FORMS)
Ug = dc.union_membership(L, FORMS, guarded=True)
Up = dc.union_membership(L, FORMS, guarded=False)
dc.write_tif(G, Fg, GEO / "F_guarded.tif")
dc.write_tif(G, Fp, GEO / "F_unguarded.tif")
dc.write_tif(G, Ug, GEO / "union_membership_guarded.tif")
for fid in FORMS:
    dc.write_tif(G, L.f_guard[fid], GEO / f"f_guarded_{fid}.tif")
print(f"wrote {2 + len(FORMS)} surfaces + union membership to {GEO.relative_to(ROOT)}")

TD2a = dc.band_table(G, {"unguarded": Fp, "guarded": Fg})
TD2a.to_csv(TAB / "T-D2_bands.csv", index=False)
print("\nT-D2 (bands, discretionary landscape):")
print(TD2a.to_string(index=False, float_format=lambda v: f"{v:.1f}"))
SUMMARY["frequent_km2"] = dict(guarded=int(((Fg >= THR) & G.disc).sum()), unguarded=int(((Fp >= THR) & G.disc).sum()))
SUMMARY["always_km2"] = dict(guarded=int(((Fg >= 0.95) & G.disc).sum()), unguarded=int(((Fp >= 0.95) & G.disc).sum()))

# Act 3 context: how much of the discretionary landscape is ever in a band
SUMMARY["ever_in_band_pct"] = dict(guarded=float(100 * (Ug[G.disc] > 0).mean()), unguarded=float(100 * (Up[G.disc] > 0).mean()))
p10 = dc.RUNS / "s0_ssp585_theta5" / "mga_g10.tif"
if p10.exists():
    u10 = ec.read_selections(p10, G.pu).any(axis=0)
    SUMMARY["s0_g10_union_pct"] = float(100 * u10[G.disc].mean())
    print(f"\never in a band: guarded {SUMMARY['ever_in_band_pct']['guarded']:.1f}% | unguarded "
          f"{SUMMARY['ever_in_band_pct']['unguarded']:.1f}% of discretionary cells; S0 alone at g=10%: "
          f"{SUMMARY['s0_g10_union_pct']:.1f}% (Gate-2b probe)")
# E11 sentence (from the Gate-4 record): ordered formulation pairs mutually within each other's 5% band
D11 = pd.read_csv(SPEC / "E11_delta_matrix.csv", index_col=0)
off = ~np.eye(len(D11), dtype=bool)
SUMMARY["e11_pairs_in_band"] = [int((D11.values[off] <= 0.05 + 1e-9).sum()), int(off.sum())]
print(f"E11: {SUMMARY['e11_pairs_in_band'][0]}/{SUMMARY['e11_pairs_in_band'][1]} ordered pairs mutually near-optimal")


In [ ]:
# ---- decision (g): climate-level pooling check; Act tiers; T-D2 act accounting ------------------
POOL, rep = dc.pool_scenarios(G, L.f_guard, MAN)
POOLp, _ = dc.pool_scenarios(G, L.f_plain, MAN)
rep.to_csv(TAB / "pooling_check.csv", index=False)
print("pooling check (frequent-tier Jaccard between climate levels; pool iff >= "
      f"{dc.POOL_JACCARD_MIN}):\n{rep.to_string(index=False, float_format=lambda v: f'{v:.3f}')}")
SUMMARY["pooling"] = rep.to_dict(orient="records")

def act_masks(Fx, POOLx, Ux):
    core = (Fx >= THR) & G.disc
    out = {"Act 1 core (F >= 0.70, all formulations)": core}
    claimed = core.copy()
    any_sc = np.zeros(G.n_pu, bool)
    for key, f in POOLx.items():
        sid = key.split("@")[0]
        tier = (f >= THR) & G.disc & ~core
        tag = "Act 2" if sid in dc.ACT2_SCENARIOS else "appendix"
        out[f"{tag} {key}: {dc.SCENARIO_LABEL[sid]} (frequent minus core)"] = tier
        if sid in dc.ACT2_SCENARIOS:
            any_sc |= tier
    out["Act 2 any named scenario (union)"] = any_sc
    claimed |= any_sc
    out["Act 3 opportunity (in >= 1 band, not above)"] = (Ux > 0) & G.disc & ~claimed
    out["never (no near-optimal plan selects it)"] = (Ux == 0) & G.disc
    return out

AG, AP = act_masks(Fg, POOL, Ug), act_masks(Fp, POOLp, Up)
rows = []
for k in AG:
    kp = AP.get(k)
    rows.append({"tier": k, "guarded km2": int(AG[k].sum()), "guarded %disc": 100 * AG[k].sum() / G.n_disc,
                 "unguarded km2": int(kp.sum()) if kp is not None else np.nan,
                 "unguarded %disc": 100 * kp.sum() / G.n_disc if kp is not None else np.nan})
TD2b = pd.DataFrame(rows)
TD2b.to_csv(TAB / "T-D2_acts.csv", index=False)
print("\nT-D2 (act tiers):")
print(TD2b.to_string(index=False, float_format=lambda v: f"{v:.1f}"))
# coded tier raster for the Act-3 map: 0 never / 1 opportunity / 2 scenario-specific / 3 core
tiers = np.zeros(G.n_pu, np.uint8)
tiers[AG["Act 3 opportunity (in >= 1 band, not above)"]] = 1
tiers[AG["Act 2 any named scenario (union)"]] = 2
tiers[AG["Act 1 core (F >= 0.70, all formulations)"]] = 3
dc.write_tif(G, tiers, GEO / "act_tiers_guarded.tif", dtype="uint8", nodata=255)


In [ ]:
# ---- clustering (pre-stated procedure) + sensitivity + top-k picks ------------------------------
named = dc.named_areas(G)
core2d = dc.to_grid(G, (Fg >= THR) & G.disc, fill=False, dtype=bool)

lab1, reg1 = dc.clusters(G, Fg)
reg1.insert(0, "act", "Act 1"); reg1.insert(1, "key", "ensemble")
reg1["name"] = [dc.placeholder_name(G, lab1 == c, named) if k else "" for c, k in zip(reg1.cid, reg1.kept)]
sens = [dc.sensitivity(G, Fg).assign(act="Act 1", key="ensemble")]
print(f"Act 1: tier {int(((Fg >= THR) & G.disc).sum()):,} km2 -> {len(reg1)} components, "
      f"{int(reg1.kept.sum())} >= {dc.MIN_KM2} km2 ({reg1[reg1.kept].km2.sum():,.0f} km2)")

LABELS = {"act1": lab1}
REGS = [reg1]
PICKS = []
number = 0
for _, r in dc.top_k(reg1, dc.TOPK_ACT1).iterrows():
    number += 1
    PICKS.append(dict(number=number, act="Act 1", key="ensemble", cid=int(r.cid), name=r["name"],
                      km2=float(r.km2), meanF=float(r.meanF), lat=float(r.lat), lon=float(r.lon)))
for key, f in POOL.items():
    sid = key.split("@")[0]
    if sid not in dc.ACT2_SCENARIOS:
        continue
    lab, reg = dc.clusters(G, f, subtract2d=core2d)
    reg.insert(0, "act", "Act 2"); reg.insert(1, "key", key)
    reg["name"] = [dc.placeholder_name(G, (lab == c) & ~core2d, named) if k else "" for c, k in zip(reg.cid, reg.kept)]
    LABELS[f"act2_{key}"] = lab
    REGS.append(reg)
    sens.append(dc.sensitivity(G, f, subtract2d=core2d).assign(act="Act 2", key=key))
    kept = reg[reg.kept]
    print(f"Act 2 {key:<8} ({dc.SCENARIO_LABEL[sid]}): {len(reg)} components, {len(kept)} kept after core "
          f"subtraction ({kept.residual_km2.sum():,.0f} km2 residual; mean core overlap of kept "
          f"{kept.core_overlap_pct.mean() if len(kept) else 0:.0f}%)")
    for _, r in kept.sort_values(["residual_km2", "meanF"], ascending=False).head(dc.TOPK_ACT2).iterrows():
        number += 1
        PICKS.append(dict(number=number, act="Act 2", key=key, cid=int(r.cid), name=r["name"],
                          km2=float(r.residual_km2), meanF=float(r.meanF), lat=float(r.lat), lon=float(r.lon)))
REG = pd.concat(REGS, ignore_index=True)
REG.to_csv(TAB / "cluster_register_all.csv", index=False)
SENS = pd.concat(sens, ignore_index=True)
SENS.to_csv(TAB / "cluster_sensitivity.csv", index=False)
PICKS = pd.DataFrame(PICKS)
PICKS.to_csv(TAB / "picks.csv", index=False)
np.savez_compressed(GEO / "cluster_labels.npz", **LABELS)
print("\nsensitivity companion (threshold 0.60 / 0.70 / 0.80):")
print(SENS.to_string(index=False, formatters={"threshold": "{:.2f}".format}, float_format=lambda v: f"{v:,.0f}"))
print("\ndeck picks (presentational top-k; register ships in full):")
print(PICKS.to_string(index=False, float_format=lambda v: f"{v:,.2f}"))

# cluster polygons (topology-preserving 2 km simplification) -> one GeoPackage, a layer per surface
gp = GEO / "clusters.gpkg"
if gp.exists():
    gp.unlink()
for k, lab in LABELS.items():
    regk = REG[(REG.key == (k.replace("act2_", "") if k != "act1" else "ensemble")) & REG.kept]
    if len(regk):
        v = dc.vectorize(G, lab, regk.cid.tolist())
        v = v.merge(regk[["cid", "name", "km2", "meanF"]], on="cid")
        v.to_file(gp, layer=k, driver="GPKG")
print(f"wrote {gp.relative_to(ROOT)} ({len(LABELS)} layers)")


In [ ]:
# ---- T-D1 cluster register (every kept cluster; the deck shows the picks) -----------------------
P = dc.block_percentiles(G)
DM = dc.driver_masks(G)
IP = dc.ipca_layer(G)
near_pa = ndimage.distance_transform_edt(~G.locked2d) <= 5     # within 5 km of an existing PA
rows = []
for _, r in REG[REG.kept].iterrows():
    lab = LABELS["act1" if r.act == "Act 1" else f"act2_{r.key}"]
    m2 = (lab == r.cid)
    if r.act == "Act 2":
        m2 = m2 & ~core2d
    m1 = m2[G.pu]
    n = int(m1.sum())
    prof = dc.star_profile(P, m1)
    pick = PICKS[(PICKS.act == r.act) & (PICKS.key == r.key) & (PICKS.cid == r.cid)]
    n_freq = int(sum(float(L.f_guard[fid][m1].mean() >= THR) >= 0.5 for fid in FORMS))  # formulations where >=50% of cells are frequent
    row = dict(number=int(pick.number.iloc[0]) if len(pick) else np.nan, name=r["name"], act=r.act,
               driving=r.key if r.act == "Act 2" else "all formulations", area_km2=n * G.cell_km2,
               mean_guarded_F=float(Fg[m1].mean()), min_guarded_F=float(Fg[m1].min()))
    row.update({f"pct_{a}": prof[a] for a in dc.STAR_AXES})
    row.update({f"driver_{k}": 100 * float(v[m1].mean()) for k, v in DM.items()})
    row.update(mean_lat=float(dc.latlon(G)[0][m1].mean()),
               pct_within_5km_of_PA=100 * float(near_pa[m2].mean()),
               pct_in_proposed_IPCA=100 * float(IP.mask2d[m2].mean()),
               n_formulations_frequent=n_freq)
    rows.append(row)
TD1 = pd.DataFrame(rows).sort_values(["act", "area_km2"], ascending=[True, False]).reset_index(drop=True)
TD1["number"] = TD1["number"].astype("Int64")
TD1.insert(4, "driving_label", [dc.SCENARIO_LABEL[d.split("@")[0]] if d != "all formulations" else "all formulations" for d in TD1.driving])
TD1.to_csv(TAB / "T-D1_cluster_register.csv", index=False)
show = ["number", "name", "act", "driving_label", "area_km2", "mean_guarded_F", "mean_lat", "pct_in_proposed_IPCA",
        "n_formulations_frequent"] + [c for c in TD1.columns if c.startswith("driver_")]
print("T-D1 (kept clusters; % columns are shares of cluster cells):")
print(TD1[show].to_string(index=False, float_format=lambda v: f"{v:,.1f}"))
print("\nNOTE clusters are discretionary by construction (PA overlap = 0), so 'pct_within_5km_of_PA' reports "
      "adjacency instead; cluster-IPCA overlap is INDEPENDENT CONVERGENCE (proposals are not locked in).")


In [ ]:
# ---- T-D3 scenario summary (from the frozen T1 record + anchors) ---------------------------------
cap = pd.read_csv(SPEC / "T1_anchor_captures.csv", index_col=0)
tail = pd.read_csv(SPEC / "T1_tail_capture.csv", index_col=0)
lat, _ = dc.latlon(G)
rows = []
for _, r in MAN.iterrows():
    fid = r.formulation_id
    if fid not in FORMS:
        continue
    row = dict(formulation=fid, scenario=dc.SCENARIO_LABEL[r.scenario_id], climate=r.climate_level.replace("_2071_2100", ""),
               value_statement=dc.SCENARIO_STATEMENT[r.scenario_id])
    for b, feats in config.BLOCKS.items():
        row[f"capture_{b}"] = float(cap.loc[fid, feats].mean())
    row["tail_m_soc"] = float(tail.loc[fid, "irrecoverable_carbon_m_soc"])
    row["tail_biomass"] = float(tail.loc[fid, "irrecoverable_carbon_biomass"])
    row["anchor_mean_lat"] = float(lat[L.anchors[fid] & G.disc].mean())
    row["frequent_km2_guarded"] = int((L.f_guard[fid][G.disc] >= THR).sum())
    row["frequent_km2_unguarded"] = int((L.f_plain[fid][G.disc] >= THR).sum())
    row["D_unguarded"], row["D_guarded"] = L.D_plain.get(fid, np.nan), L.D_guard.get(fid, np.nan)
    rows.append(row)
TD3 = pd.DataFrame(rows)
TD3.to_csv(TAB / "T-D3_scenarios.csv", index=False)
print("T-D3 (block captures = mean captured fraction of the block's features; tails = theta-tail mass capture):")
print(TD3.drop(columns="value_statement").to_string(index=False, float_format=lambda v: f"{v:,.3f}"))


In [ ]:
# ---- v1.3 additions: tier-achievement (zero-solve) + T-D4 tier area by ecoregion ---------------
core1 = AG["Act 1 core (F >= 0.70, all formulations)"]
sc1 = AG["Act 2 any named scenario (union)"]
opp1 = AG["Act 3 opportunity (in >= 1 band, not above)"]
CUM = {"existing PAs": G.locked, "+ Act 1 core": G.locked | core1,
       "+ Act 2 scenario tiers": G.locked | core1 | sc1, "+ Act 3 opportunity": G.locked | core1 | sc1 | opp1}
TA = dc.tier_achievement(G, CUM)
# anchor-level reference: block captures of every anchor (T1 record) -> S0 + min/max across formulations
ref = TD3[[c for c in TD3.columns if c.startswith("capture_")]].rename(columns=lambda c: c.replace("capture_", ""))
ref.index = TD3.formulation
TA_ref = pd.DataFrame({"s0": ref.loc["s0_ssp585_theta5"] if "s0_ssp585_theta5" in ref.index else ref.iloc[0],
                       "anchor_min": ref.min(), "anchor_max": ref.max()})
TA.to_csv(TAB / "tier_achievement.csv", index=False); TA_ref.to_csv(TAB / "tier_achievement_reference.csv")
piv = TA[TA.feature == "BLOCK"].pivot(index="tier", columns="block", values="capture").reindex(list(CUM))
print("tier achievement (block capture, cumulative tiers incl. locked PAs):")
print(piv.to_string(float_format=lambda v: f"{v:.3f}"))
print("anchor reference (S0 / min / max across formulations):")
print(TA_ref.T.to_string(float_format=lambda v: f"{v:.3f}"))
SUMMARY["tier_area_pct_disc"] = {k: float(100 * (m & G.disc).sum() / G.n_disc) for k, m in
                                 [("core", core1), ("scenario", sc1), ("opportunity", opp1)]}

cap = pd.read_csv(SPEC / "T1_anchor_captures.csv", index_col=0)
# T-D5 protected baseline: what the existing PA estate already banks of each value (M3.6 accounting:
# PAs are locked in, so banked amounts COUNT toward targets and the 30% budget INCLUDES PA area)
sc0 = json.loads((SPEC / "scenarios_v2.json").read_text())["S0_balanced"]["targets"]
pa_area_share = float(G.locked.sum() / G.n_pu)
rows = []
for f in lc.continuous_features():
    v = np.nan_to_num(lc._read(config.HANDOFF_DIR / f"{f}.tif")[G.pu], nan=0.0)
    pa = float(v[G.locked].sum() / v.sum()); tgt = float(sc0.get(f, 1.0))
    s0cap = float(cap.loc["s0_ssp585_theta5", f]) if "s0_ssp585_theta5" in cap.index else np.nan
    rows.append(dict(value=f, pct_of_regional_total_in_PAs=100 * pa, S0_target=tgt,
                     pct_of_target_already_banked=100 * pa / tgt,
                     pct_still_needed_from_unprotected_land=100 * max(tgt - pa, 0),
                     enrichment_existing_PAs=pa / pa_area_share,                       # capture share / area share
                     enrichment_S0_new_half=(s0cap - pa) / (config.BUDGET_PCT - pa_area_share)))  # the optimizer's 15%
TD5 = pd.DataFrame(rows)
TD5.to_csv(TAB / "T-D5_protected_baseline.csv", index=False)
n_efg_pa = int(P.efg[:, G.locked].any(axis=1).sum())
SUMMARY["protected_baseline"] = dict(pa_km2=int(G.locked.sum()), pa_pct_of_region=float(100 * G.locked.sum() / G.n_pu),
                                     pa_pct_of_budget=float(100 * G.locked.sum() / (config.BUDGET_PCT * G.n_pu)),
                                     efg_present_in_PAs=n_efg_pa, banked_min=float(TD5.pct_of_regional_total_in_PAs.min()),
                                     banked_max=float(TD5.pct_of_regional_total_in_PAs.max()))
print("T-D5 protected baseline (existing PAs, locked in every plan; enrichment = capture share / area share):")
print(TD5.to_string(index=False, float_format=lambda v: f"{v:.1f}"))
print(f"PA estate {G.locked.sum():,} km2 = {SUMMARY['protected_baseline']['pa_pct_of_region']:.1f}% of the region = "
      f"{SUMMARY['protected_baseline']['pa_pct_of_budget']:.1f}% of the 30% budget; {n_efg_pa}/40 EFG classes present inside PAs\n")

# T-D5b enrichment by scenario: per value, capture share / area share for (i) each scenario anchor's NEW
# half (anchor capture minus the PA-banked share, over the 15% the optimizer chose) and (ii) each
# scenario's guarded FREQUENT tier (f >= 0.70, discretionary) plus the guarded ensemble core -- what the
# package promises. Anchors from the frozen T1 record (all 14); tiers from the guarded sweeps present.
VALS = {f: np.nan_to_num(lc._read(config.HANDOFF_DIR / f"{f}.tif")[G.pu], nan=0.0) for f in lc.continuous_features()}
banked = {f: float(v[G.locked].sum() / v.sum()) for f, v in VALS.items()}
def tier_enrich(mask1d):
    a = mask1d.sum() / G.n_pu
    return {f: float(v[mask1d].sum() / v.sum()) / a if a > 0 else np.nan for f, v in VALS.items()}
E5 = {}
for _, r in MAN.iterrows():
    fid = r.formulation_id
    lab = f"{dc.SCENARIO_LABEL[r.scenario_id].split(' (')[0]} {'585' if 'ssp585' in fid else '245'}"
    E5[f"anchor · {lab}"] = {f: (float(cap.loc[fid, f]) - banked[f]) / (config.BUDGET_PCT - pa_area_share) for f in VALS}
    if fid in FORMS:
        E5[f"frequent tier · {lab}"] = tier_enrich((L.f_guard[fid] >= THR) & G.disc)
E5["frequent tier · ENSEMBLE core"] = tier_enrich((Fg >= THR) & G.disc)
E5["existing PAs"] = {f: banked[f] / pa_area_share for f in VALS}
TD5b = pd.DataFrame(E5)
TD5b.index.name = "value"
TD5b.to_csv(TAB / "T-D5b_enrichment_by_scenario.csv")
print("T-D5b enrichment (capture share / area share) -- anchors' new half vs guarded frequent tiers:")
show = [c for c in TD5b.columns if c.startswith("existing") or "585" in c or "ENSEMBLE" in c]
print(TD5b[show].T.to_string(float_format=lambda v: f"{v:.2f}"))

ECO = dc.ecoregion_layer(G)
if ECO is None:
    print(f"\nT-D4 PENDING: no ecozone/ecoregion vector in {dc.ECOREGIONS_DIR.relative_to(ROOT)}/ -- drop one there "
          "(e.g. CEC North American Level II/III ecoregions, seamless US+Canada) and re-run this cell")
    SUMMARY["td4"] = "pending (no ecoregion layer)"
else:
    z = ECO.zones[G.pu]
    tiers1 = np.zeros(G.n_pu, np.uint8); tiers1[opp1] = 1; tiers1[sc1] = 2; tiers1[core1] = 3
    rows = []
    for _, zr in ECO.gdf.iterrows():
        inz = z == zr.zone_id
        if inz.sum() == 0:
            continue
        rows.append({"ecoregion": zr[ECO.name_field], "PU km2": int(inz.sum()), "protected km2": int((inz & G.locked).sum()),
                     "core km2": int((inz & (tiers1 == 3)).sum()), "scenario km2": int((inz & (tiers1 == 2)).sum()),
                     "opportunity km2": int((inz & (tiers1 == 1)).sum()), "never km2": int((inz & G.disc & (tiers1 == 0)).sum()),
                     "mean lat": float(dc.latlon(G)[0][inz].mean())})
    TD4 = pd.DataFrame(rows).sort_values("core km2", ascending=False)
    TD4.to_csv(TAB / "T-D4_ecoregions.csv", index=False)
    SUMMARY["td4"] = f"from {ECO.source} ({ECO.name_field})"
    print(f"\nT-D4 (tier area by {ECO.name_field}, {ECO.source}):"); print(TD4.to_string(index=False))


In [ ]:
# ---- E17 inputs for the one-pager + summary.json ------------------------------------------------
base_lat, E17 = dc.e17_shifts(G)
E17.to_csv(TAB / "E17_shifts.csv", index=False)
geo = pd.read_csv(SPEC / "e17_efg_geography.csv")
n_south = int((geo.south_share > 0.90).sum())
SUMMARY["e17"] = dict(base_lat=base_lat, n_efg_south=n_south, n_efg=len(geo), median_efg_lat=float(geo.mean_lat.median()),
                      shifts=E17.to_dict(orient="records"))
print(f"E17: S0 anchor mean latitude {base_lat:.2f}N; {n_south}/{len(geo)} EFGs >90% south of 53N; shifts:\n"
      f"{E17.to_string(index=False, float_format=lambda v: f'{v:+.2f}')}")
SUMMARY["forms"] = FORMS
SUMMARY["pool_keys"] = list(POOL)
(PKG / "summary.json").write_text(json.dumps(SUMMARY, indent=1, default=float))
print(f"\nwrote {PKG.relative_to(ROOT)}/summary.json -- next: 20_director_figures.ipynb")
